## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: Union[str, ForwardRef('os.PathLike[str]'), NoneType] = None, stream: Optional[IO[str]] = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: Optional[str] = 'utf-8') -> bool>

In [ ]:
groq_api_key = os.getenv("GROQ_API_KEY")

In [3]:
from langchain_groq import ChatGroq
model = ChatGroq(model="openai/gpt-oss-20b", groq_api_key=groq_api_key)
model

d:\Learning\Generative AI\Practicals\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001DAF7CBDD20>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001DAF7CBE410>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [4]:
from langchain_core.messages import HumanMessage
model.invoke([
    HumanMessage(content="Hi, My name is Sidhant and I am a AI engineer")
])

AIMessage(content='Hello Sidhant! 👋 It’s great to meet another AI engineer. How can I assist you today? Whether you’re looking for insights on a specific project, want to brainstorm ideas, or just curious about the latest in AI research, I’m here to help!', additional_kwargs={'reasoning_content': 'User introduction: "Hi, My name is Sidhant and I am a AI engineer". They likely want a response. We should be friendly, ask what they need. Possibly mention how can we help.'}, response_metadata={'token_usage': {'completion_tokens': 107, 'prompt_tokens': 85, 'total_tokens': 192, 'completion_time': 0.134355911, 'completion_tokens_details': {'reasoning_tokens': 43}, 'prompt_time': 0.004243636, 'prompt_tokens_details': None, 'queue_time': 0.052121994, 'total_time': 0.138599547}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e99e93f2ac', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--01a1e789-d9d5-7681-92c9-671c3b8f324c-0', usage_metadata={'input

In [5]:
from langchain_core.messages import AIMessage

model.invoke([
    HumanMessage(content="Hi my name is Sidhant and I like to listen music and im a software engineer"),
    AIMessage(content="Hello Sidhant! 👋 It’s great to meet an AI engineer. How can I help you today? Whether you’re working on models, data pipelines, or just curious about the latest in AI, I’m here for you."),
    HumanMessage(content = "Hey what is my name and what I do")
])

AIMessage(content='You’re Sidhant, a software engineer who enjoys listening to music.', additional_kwargs={'reasoning_content': 'The user says: "Hey what is my name and what I do". They want a reminder: name is Sidhant, they like to listen to music and are a software engineer. Should answer accordingly.'}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 156, 'total_tokens': 223, 'completion_time': 0.0785134, 'completion_tokens_details': {'reasoning_tokens': 43}, 'prompt_time': 0.011008231, 'prompt_tokens_details': None, 'queue_time': 0.314492478, 'total_time': 0.089521631}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_d6de37e6be', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--01a1e789-dd93-7729-873b-11abd7647384-0', usage_metadata={'input_tokens': 156, 'output_tokens': 67, 'total_tokens': 223})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [6]:
from langchain_community.chat_message_histories import ChatMessageHistory ## stores the message in the list
from langchain_core.chat_history import BaseChatMessageHistory ## abstract class means fo inhertance to define return type.
from langchain_core.runnables.history import RunnableWithMessageHistory

"""
## RunnableWithMessageHistory
Loads the past messages for a given session before each call
Injects them into the prompt
Saves the new human message and AI reply back to the history afterward

"""

store = {}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]    

with_message_history = RunnableWithMessageHistory(model, get_session_history)

d:\Learning\Generative AI\Practicals\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [7]:
config1 = {"configurable":{"session_id":"Chat1"}}

In [8]:
reponse = with_message_history.invoke(
[HumanMessage(content="Hi, My name is Alice and I am a cook")],
config = config1
)

In [9]:
with_message_history.invoke(
    [HumanMessage(content="what is my name?")],
    config = config1
)

AIMessage(content='Your name is Alice.', additional_kwargs={'reasoning_content': 'User: "what is my name?" They previously said: "Hi, My name is Alice and I am a cook". So answer: Alice.'}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 126, 'total_tokens': 171, 'completion_time': 0.047415233, 'completion_tokens_details': {'reasoning_tokens': 31}, 'prompt_time': 0.007028238, 'prompt_tokens_details': None, 'queue_time': 0.255237161, 'total_time': 0.054443471}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_976a85d17a', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--01a1e789-e185-77fc-b255-d8fa1f7a7302-0', usage_metadata={'input_tokens': 126, 'output_tokens': 45, 'total_tokens': 171})

In [10]:
with_message_history.invoke(
    [HumanMessage(content="what is my name and what do i do?")],
    config = config1
)

AIMessage(content='You’re Alice, and you’re a cook—so you’re the one whipping up delicious meals and experimenting with flavors in the kitchen. If you want to share more about your cooking style or ask for recipes, just let me know!', additional_kwargs={'reasoning_content': 'User: "what is my name and what do i do?" We have conversation: user said "Hi, My name is Alice and I am a cook". So we know name is Alice, occupation cook. Should respond accordingly.'}, response_metadata={'token_usage': {'completion_tokens': 103, 'prompt_tokens': 151, 'total_tokens': 254, 'completion_time': 0.109526848, 'completion_tokens_details': {'reasoning_tokens': 47}, 'prompt_time': 0.007476039, 'prompt_tokens_details': None, 'queue_time': 0.364854917, 'total_time': 0.117002887}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_bb6a356524', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--01a1e789-e3b0-74d3-8eba-b3e370e915a1-0', usage_metadata={'input_tokens': 1

In [11]:
config2 = {"configurable" : {"session_id" : "chat2"}}


In [12]:
with_message_history.invoke(
    [HumanMessage(content="what is my name")],
    config = config2
)

AIMessage(content='I don’t have that information—could you let me know what you’d like me to call you?', additional_kwargs={'reasoning_content': 'User: "what is my name". They likely want to know their name. There\'s no context. According to policy: no personal data. The user is asking for personal info. We must not provide personal data. We should respond with "I don\'t know your name" or ask them to tell us. So we respond: "I don\'t know your name. Please tell me."'}, response_metadata={'token_usage': {'completion_tokens': 107, 'prompt_tokens': 75, 'total_tokens': 182, 'completion_time': 0.171877784, 'completion_tokens_details': {'reasoning_tokens': 77}, 'prompt_time': 0.003581255, 'prompt_tokens_details': None, 'queue_time': 0.315968095, 'total_time': 0.175459039}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e189667b30', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--01a1e789-e648-713d-96f9-797b682def3a-0', usage_metadata={'input_

In [13]:
with_message_history.invoke(
    [HumanMessage(content="what is Eifil tower")],
    config = config2
)

AIMessage(content='**Eiffel Tower** (often misspelled “Eifil” or “Eifel”) is one of the world’s most iconic landmarks, located in Paris, France. Here’s a quick snapshot of what makes it special:\n\n| Feature | Details |\n|---------|---------|\n| **Location** | Champ de Mars, Paris, France |\n| **Height** | 330\u202fm (1,083\u202fft) – including its antennas |\n| **Construction** | 1887‑1889, designed by engineer **Gustave Eiffel** (hence the name) |\n| **Purpose** | Originally built as the entrance arch for the 1889 World’s Fair (Exposition Universelle) |\n| **Structure** | 18,038 steel parts, 2.5\u202fmillion rivets, 300\u202fm of iron |\n| **Visitors** | ~7–8\u202fmillion per year (2023) |\n| **Floors** | Three public levels (restaurants, shops, observation decks) + a radio‑broadcasting tower |\n| **Notable Facts** |\n| • The tower was almost dismantled after the fair but became a symbol of French ingenuity. |\n| • It was the tallest man‑made structure in the world until 1930. |\n| •

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [14]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant.Amnswer all the question to the nest of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt|model

In [15]:
chain.invoke({"messages" : [HumanMessage(content = "Hi my name is Alice")]})

AIMessage(content='Hello Alice! 👋 How can I help you today?', additional_kwargs={'reasoning_content': 'User says "Hi my name is Alice". They just introduced themselves. We should respond politely, ask how we can help.'}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 97, 'total_tokens': 144, 'completion_time': 0.063240617, 'completion_tokens_details': {'reasoning_tokens': 26}, 'prompt_time': 0.016388609, 'prompt_tokens_details': None, 'queue_time': 0.25563307, 'total_time': 0.079629226}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_334cc21c60', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--01a1e789-ee7f-7257-bee4-8e653149f89e-0', usage_metadata={'input_tokens': 97, 'output_tokens': 47, 'total_tokens': 144})

In [16]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

d:\Learning\Generative AI\Practicals\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [17]:
config3 = {"configurable" : {"session_id" : "chat3"}}

response = with_message_history.invoke(
    [HumanMessage(content="Hi my name is Alice")],
    config = config3
)

In [18]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config3,
)

In [19]:
response

AIMessage(content='You’re Alice!', additional_kwargs={'reasoning_content': 'User says "Hi my name is Alice". They ask "What\'s my name?" The answer: Alice. Should respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 123, 'total_tokens': 162, 'completion_time': 0.044266353, 'completion_tokens_details': {'reasoning_tokens': 26}, 'prompt_time': 0.006092023, 'prompt_tokens_details': None, 'queue_time': 0.047793157, 'total_time': 0.050358376}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e99e93f2ac', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--01a1e789-f386-7107-85dc-07717f485ecb-0', usage_metadata={'input_tokens': 123, 'output_tokens': 39, 'total_tokens': 162})

In [20]:
response.content

'You’re Alice!'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [21]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [22]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Sidhant")],"language":"Hindi"})
response.content

'नमस्ते सिद्धांत जी! आपका स्वागत है। मैं आपकी किस प्रकार सहायता कर सकता हूँ?'

In [23]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

d:\Learning\Generative AI\Practicals\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [24]:
config = {"configurable": {"session_id": "chat4"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Sidhant")],"language":"Hindi"},
    config=config
)
repsonse.content

'नमस्ते सिद्धांत जी! 👋 मैं आपकी किस प्रकार सहायता कर सकता हूँ?'

In [25]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Hindi"},
    config=config,
)

In [26]:
response.content

'आपका नाम सिद्धांत है।'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [44]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45, ## decrease it to low number like 45 to see how are model forgets the some previous history
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

## output - when max_token=45 or less then , the messages after the system message has been deleted.

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={})]

In [45]:
## now chain with trimmer
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

'I’m not sure—do you have a favorite flavor or brand you usually go for? Let me know and I’ll try to guess or suggest something you might enjoy!'

In [46]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'You asked: **“What’s 2\u202f+\u202f2?”**'

In [47]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

d:\Learning\Generative AI\Practicals\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [48]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

##Note: the AI should respond like the AI dont know about the question but as we have used "message1" which creates the problem and generates the content like below. 

'I’m not sure what your name is—could you let me know?'